# Astrology input correlation screen

Public project: [CastaliaInstitute/solar-revolutions](https://github.com/CastaliaInstitute/solar-revolutions)  
Public page: [castaliainstitute.github.io/solar-revolutions](https://castaliainstitute.github.io/solar-revolutions/)

## Question

Do any of our historical inputs correlate with the exploratory astrology inputs?

## Scientific posture

The astrology inputs are exploratory/placebo-style cyclic proxies. This notebook is designed to find correlations, but also to make them hard to overstate by reporting overlap, lag sensitivity, multiple-comparison pressure, and circular-shift placebo checks. Correlation here is not causal evidence.


## 1. Hypotheses

**H0:** astrology proxy inputs do not meaningfully correlate with historical inputs beyond chance, shared trends, autocorrelation, or multiple testing.

**H1:** one or more astrology proxy inputs show unusually strong and robust association with historical inputs after transparent screening, lag checks, and placebo-cycle comparison.

A result is interesting only if it has adequate overlap, survives multiple-comparison pressure, and beats circularly shifted versions of the same astrology proxy.


## Current executed readout

Latest local run found `528` same-year astrology-input pairs across six astrology proxies and 89 non-astrology catalog inputs. The strongest raw same-year pair was `astrology.uranus_pluto_hard_aspect_proximity` vs `resistance.navco_injury_proxy` with `r = 0.637` over `n = 23` years, but that pair did **not** beat its own circular-shift placebo at `p < 0.05` (`p ≈ 0.091`).

Among the top 25 raw same-year pairs, `7` beat the circular-shift placebo threshold. These are exploratory candidates only, not evidence for astrology as a causal mechanism. The cleaner interpretation is that some astrology proxies behave like cyclic/time-structured placebo variables that can align with short historical series.

Current skeptical verdict: several top pairs beat circular-shift placebo, so the correlations are worth browsing, but any follow-up must be preregistered, tested out of sample, and compared against ordinary sine/cycle controls and conventional historical predictors before interpretation.


## 2. Setup

Run this first. In Colab, the public repository is cloned if needed.


In [ ]:
#@title Install notebook dependencies { display-mode: "form" }
INSTALL_PACKAGES = False

if INSTALL_PACKAGES:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2,<3", "numpy>=1.26,<3", "scipy>=1.11,<2"], check=True)


In [ ]:
#@title Locate or clone public repository { display-mode: "form" }
import json
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/CastaliaInstitute/solar-revolutions.git"

def running_in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def api_root_for(candidate: Path):
    for api in [candidate / "api", candidate / "dashboard" / "public" / "api"]:
        if (api / "input-series.json").exists() and (api / "input-data").exists():
            return api
    return None

def find_repo():
    env_root = os.environ.get("SOLAR_REPO_ROOT") or os.environ.get("APOCALYPSO_REPO_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents, Path("/content/solar-revolutions")])
    for candidate in candidates:
        api = api_root_for(candidate)
        if api is not None:
            return candidate, api
    if running_in_colab():
        target = Path("/content/solar-revolutions")
        if not target.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
        api = api_root_for(target)
        if api is not None:
            return target, api
    raise FileNotFoundError("Could not find public API artifacts. Set SOLAR_REPO_ROOT to the solar-revolutions checkout.")

ROOT, API = find_repo()
INPUT_DATA = API / "input-data"
print("Repo root:", ROOT)
print("API:", API)


## 3. Load and annualize catalog inputs


In [ ]:
#@title Load annual catalog inputs { display-mode: "form" }
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def parse_year(value):
    try:
        return int(str(value)[:4])
    except Exception:
        return None

def series_path(series_id):
    return INPUT_DATA / f"{series_id.replace('.', '_')}.json"

def annualize(series_id):
    data = load_json(series_path(series_id))
    buckets = {}
    for point in data.get("points", []):
        year = point.get("year") if isinstance(point.get("year"), int) else parse_year(point.get("date"))
        try:
            value = float(point.get("value"))
        except Exception:
            continue
        if year is None or not np.isfinite(value):
            continue
        buckets.setdefault(year, []).append(value)
    return pd.Series({year: float(np.mean(values)) for year, values in buckets.items()}, name=series_id).sort_index()

catalog = load_json(API / "input-series.json")["series"]
catalog_by_id = {item["id"]: item for item in catalog}
astrology_ids = [item["id"] for item in catalog if item["id"].startswith("astrology.") and series_path(item["id"]).exists()]
candidate_ids = [item["id"] for item in catalog if not item["id"].startswith("astrology.") and series_path(item["id"]).exists()]
series = {sid: annualize(sid) for sid in [*astrology_ids, *candidate_ids]}
print("Astrology inputs:", len(astrology_ids), astrology_ids)
print("Candidate non-astrology inputs:", len(candidate_ids))


## 4. Same-year all-vs-astrology correlation screen

This ranks every non-astrology input against every astrology proxy over overlapping annual observations.


In [ ]:
#@title Same-year Pearson/Spearman correlations { display-mode: "form" }
def pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3 or x.std() == 0 or y.std() == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def spearman(x, y):
    return pearson(pd.Series(x).rank().to_numpy(), pd.Series(y).rank().to_numpy())

rows = []
for astro_id in astrology_ids:
    a = series[astro_id]
    for candidate_id in candidate_ids:
        b = series[candidate_id]
        years = sorted(set(a.index).intersection(b.index))
        if len(years) < 10:
            continue
        x = [a.loc[year] for year in years]
        y = [b.loc[year] for year in years]
        r = pearson(x, y)
        rho = spearman(x, y)
        rows.append({
            "astrology_id": astro_id,
            "astrology_name": catalog_by_id.get(astro_id, {}).get("name", astro_id),
            "input_id": candidate_id,
            "input_name": catalog_by_id.get(candidate_id, {}).get("name", candidate_id),
            "n": len(years),
            "start": years[0],
            "end": years[-1],
            "pearson_r": r,
            "spearman_rho": rho,
            "abs_pearson": abs(r) if np.isfinite(r) else np.nan,
            "abs_spearman": abs(rho) if np.isfinite(rho) else np.nan,
        })
same_year_results = pd.DataFrame(rows).sort_values("abs_pearson", ascending=False)
same_year_results.head(30)


## 5. Lag scan

Lag scans are useful for discovery but increase multiple-testing risk. Treat them as candidate generation only. Positive lag means the astrology proxy is shifted earlier relative to the input outcome.


In [ ]:
#@title Lagged astrology-input correlations { display-mode: "form" }
LAGS = list(range(-11, 12))
lag_rows = []
for astro_id in astrology_ids:
    a = series[astro_id]
    for candidate_id in candidate_ids:
        b = series[candidate_id]
        best = None
        for lag in LAGS:
            shifted = a.copy()
            shifted.index = shifted.index + lag
            years = sorted(set(shifted.index).intersection(b.index))
            if len(years) < 10:
                continue
            r = pearson([shifted.loc[year] for year in years], [b.loc[year] for year in years])
            row = {"astrology_id": astro_id, "input_id": candidate_id, "lag": lag, "n": len(years), "pearson_r": r, "abs_pearson": abs(r) if np.isfinite(r) else np.nan}
            if best is None or row["abs_pearson"] > best["abs_pearson"]:
                best = row
        if best is not None:
            best.update({"astrology_name": catalog_by_id.get(astro_id, {}).get("name", astro_id), "input_name": catalog_by_id.get(candidate_id, {}).get("name", candidate_id)})
            lag_rows.append(best)
lag_results = pd.DataFrame(lag_rows).sort_values("abs_pearson", ascending=False)
lag_results.head(30)


## 6. Multiple-comparison and placebo-shift checks

This checks whether top same-year correlations are unusual compared with circularly shifted versions of the same astrology series.


In [ ]:
#@title Circular-shift placebo checks for top pairs { display-mode: "form" }
TOP_N = 25
placebo_rows = []
for _, row in same_year_results.head(TOP_N).iterrows():
    astro = series[row["astrology_id"]]
    candidate = series[row["input_id"]]
    years = sorted(set(astro.index).intersection(candidate.index))
    x = np.asarray([astro.loc[year] for year in years], dtype=float)
    y = np.asarray([candidate.loc[year] for year in years], dtype=float)
    observed = pearson(x, y)
    shifted_abs = []
    for shift in range(1, len(x)):
        shifted_abs.append(abs(pearson(np.roll(x, shift), y)))
    p_two_sided = float(np.mean(np.asarray(shifted_abs) >= abs(observed))) if shifted_abs else np.nan
    placebo_rows.append({
        "astrology_id": row["astrology_id"],
        "astrology_name": row["astrology_name"],
        "input_id": row["input_id"],
        "input_name": row["input_name"],
        "n": int(row["n"]),
        "observed_r": observed,
        "abs_observed_r": abs(observed),
        "circular_shift_p_two_sided": p_two_sided,
        "beats_shift_p05": p_two_sided < 0.05,
    })
placebo_results = pd.DataFrame(placebo_rows).sort_values(["beats_shift_p05", "abs_observed_r"], ascending=[False, False])
placebo_results


## 7. Family-level summary

This asks which astrology proxy, if any, has the strongest overall correlation profile across the catalog.


In [ ]:
#@title Summarize correlation profile by astrology input { display-mode: "form" }
family_summary = same_year_results.groupby(["astrology_id", "astrology_name"]).agg(
    pairs=("input_id", "count"),
    median_abs_r=("abs_pearson", "median"),
    p90_abs_r=("abs_pearson", lambda x: float(np.nanpercentile(x, 90))),
    max_abs_r=("abs_pearson", "max"),
).reset_index().sort_values("max_abs_r", ascending=False)
family_summary


## 8. Computed verdict

This cell states the skeptical readout from the current notebook run.


In [ ]:
#@title Computed skeptical verdict { display-mode: "form" }
top = same_year_results.iloc[0] if len(same_year_results) else None
placebo_pass = int(placebo_results["beats_shift_p05"].sum()) if len(placebo_results) else 0
total_pairs = int(len(same_year_results))
top_text = "none" if top is None else f"{top.astrology_id} vs {top.input_id}: r={top.pearson_r:.3f}, n={int(top.n)}"
if placebo_pass == 0:
    verdict = "No top astrology-input pair beats the circular-shift placebo at p<0.05; treat correlations as exploratory artifacts until stronger evidence appears."
elif placebo_pass <= 2:
    verdict = "A small number of top pairs beat circular-shift placebo; treat as candidates only and validate out of sample."
else:
    verdict = "Several top pairs beat circular-shift placebo; inspect mechanisms and preregister follow-up tests before making claims."
print("Total same-year pairs:", total_pairs)
print("Top same-year pair:", top_text)
print("Top-25 pairs beating circular-shift p<0.05:", placebo_pass)
print("Verdict:", verdict)


## 9. Next steps

If a pair looks interesting, do not stop at this screen. Next steps are: preregister the pair, test on a held-out period, compare against non-astrology sine/cycle controls, add conventional historical controls, and define a plausible mechanism before interpreting it.
